## 依留言日期對應有效 PERMNO
使用 00 的 `Ticker_name.txt`，逐筆判斷 `secinfostartdt <= comment_date <= secinfoenddt`，起訖日皆包含。`comment_date` 是留言 Unix timestamp 的 UTC 日期，不是 thread 日期或月份。

- 當天有效的 **不同 PERMNO** 恰好一個：`matched`，輸出 ticker 與 PERMNO。
- 零個：`unmatched`；多於一個：`ambiguous`，不直接配對。
- 同一 PERMNO 的重複或重疊歷史列只算一個候選 PERMNO。
- 無效留言日期標為 `unmatched`，原因 `invalid_comment_date`。
- 不改動大小寫、token 邊界、ticker 字典範圍及原始留言不去重的規則。
- 每個來源檔新增 `*.ticker_mapping_audit.parquet`，保存全部文字候選 ticker、日期、狀態、原因及有效候選 PERMNO；只有 `matched` 進入 comments、mentions 與 long format。
- comment-level 的 `matched_tickers` 與 `matched_permnos` 是等長、位置對齊的清單；long format 保留 `permno`，避免下游再以 ticker 猜測證券身分。
- 每次執行建立新的 run，既有資料不改寫。02_01／02_02 的來源與人工審核數量須在新 matching run 上重新核對；既有 04 與 SNA 不會自動重建。


# Reddit Ticker Matching

將 01 階段清理後的 6 份 CSV 分塊處理，依 ticker 字典比對留言中的股票代碼，未匹配成功的留言刪除。

## 比對規則

- 使用成功的 00 階段 ticker 字典。
- 比對不區分大小寫。
- Token 定義為連續的 Unicode `\w+`：
  - `$AAPL`、`AAPL,`、`AAPL.` 可匹配。
  - `XAAPL`、`AAPL2`、`AAPL_test` 不匹配。
- 同一留言中的同一 ticker 只記錄一次，並按字母排序。
- 不對留言去重，即使 `id` 重複也保留。

## 資料處理

- 全程分塊處理。
- 跨 chunk 的唯一值統計使用 SQLite。
- 不使用逐列 × ticker dictionary 的 Python 迴圈。
- Unix timestamp 以 UTC 轉換日期相關欄位。
- 無法解析的日期無法對應有效期間；文字候選記入 audit，標為 unmatched。

## Comment Enrichment

從原始 cleaned CSV 保留並整理：

- `id`
- `comment_id`
- `parent_id`
- `thread_id`
- UTC timestamp / date
- `year_month`
- `matched_tickers` 與位置對齊的 `matched_permnos`

空白 thread 維持 null；只有具有效日期及唯一 PERMNO 的 ticker 才是成功匹配。

## 輸出

- 有匹配 ticker 的 comment-level Parquet
- ticker mention / pair Parquet
- `run_summary.json`

未匹配留言不寫入 Parquet，但納入 diagnostics。

每次執行建立新的 UTC 時間戳資料夾；成功後更新 `pipeline_runs.json`，失敗則不覆蓋既有 pointer。舊輸出保留。

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import csv, hashlib, importlib.metadata, json, platform, re, sqlite3, time, uuid
from functools import lru_cache
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'Data/process').is_dir())

import sys
sys.path.insert(0,str(ROOT/'Notebook'))
from pipeline_io import new_run, publish, input_run, remove_work_database

INPUT_DIR = input_run(ROOT,'cleaned')
# Explicitly pin the successful new 00 run; never fall back to the approximate universe.
TICKER_RUN = ROOT/'Data/process/00_biild_ticker_list/2026-10-05_08-12-26_UTC'
DICTIONARY = TICKER_RUN/'Ticker_name.txt'
PERMNO_FILE = TICKER_RUN/'PERMNO_list.txt'
TICKER_SUMMARY = json.loads((TICKER_RUN/'run_summary.json').read_text(encoding='utf-8'))
assert TICKER_SUMMARY['status']=='complete'
SOURCES = [INPUT_DIR / f'wallstreetbets_comment_part{i}.csv' for i in range(1, 7)]
assert all(p.is_file() for p in SOURCES), 'Expected exactly six named input files'
CHUNK_SIZE = 50_000
COLUMNS = ['id', 'parent_id', 'author', 'created_utc', 'link_id', 'body']
RUN_DIR = new_run(ROOT,'matching')
INPUT_STATS = {str(p): (p.stat().st_size, p.stat().st_mtime_ns) for p in SOURCES}
VERSIONS = {p: importlib.metadata.version(p) for p in ['pandas', 'pyarrow']}
print('Output:', RUN_DIR)
print('Versions:', VERSIONS)
display(pd.DataFrame([{'file': p.name, 'bytes': p.stat().st_size} for p in SOURCES]))

def read_chunks(path, nrows=None):
    return pd.read_csv(path, usecols=COLUMNS, dtype=str, keep_default_na=False,
                       na_filter=False, encoding='utf-8-sig', chunksize=CHUNK_SIZE,
                       nrows=nrows, on_bad_lines='error')

DICTIONARY_BYTES = DICTIONARY.read_bytes()
PERMNO_BYTES = PERMNO_FILE.read_bytes()
ticker_history = pd.read_csv(DICTIONARY,sep='\t',dtype=str,keep_default_na=False,encoding='utf-8-sig')
assert set(['ticker','secinfostartdt','secinfoenddt','permno']).issubset(ticker_history.columns)
ticker_history['ticker']=ticker_history.ticker.str.strip().str.upper()
assert ticker_history.ticker.str.len().gt(2).all()
history_start=pd.to_datetime(ticker_history.secinfostartdt,utc=True,errors='raise')
history_end=pd.to_datetime(ticker_history.secinfoenddt,utc=True,errors='raise')
assert (history_start<=history_end).all()
assert ((history_start<=pd.Timestamp('2022-12-31',tz='UTC')) & (history_end>=pd.Timestamp('2012-01-01',tz='UTC'))).all()
assert ticker_history.permno.str.fullmatch(r'[0-9]+').all()
assert pd.to_numeric(ticker_history.permno).gt(0).all()
assert set(ticker_history.permno)==set(PERMNO_FILE.read_text(encoding='utf-8').splitlines())
tickers=sorted(set(ticker_history.ticker))
assert tickers and all(re.fullmatch(r'[A-Z]+',t) for t in tickers), 'Review tokenization for non-alphabetic tickers'
TICKER_SET=frozenset(tickers)
print({'dictionary_rows':len(ticker_history),'unique_tickers':len(tickers),'unique_permnos':ticker_history.permno.nunique(),
       'date_min':history_start.min().isoformat(),'date_max':history_end.max().isoformat(),
       'unique_authors':'N/A','unique_threads':'N/A','source':str(DICTIONARY)})


Output: c:\Users\eagle\RedditStock\Data\process\02_ticker_matching\2026-10-15_02-43-43_UTC
Versions: {'pandas': '3.0.5', 'pyarrow': '22.0.0'}


,file,bytes
0,wallstreetbets_comment_part1.csv,62042383
1,wallstreetbets_comment_part2.csv,94936845
2,wallstreetbets_comment_part3.csv,353797223
3,wallstreetbets_comment_part4.csv,540772868
4,wallstreetbets_comment_part5.csv,668548705
5,wallstreetbets_comment_part6.csv,331424719


{'dictionary_rows': 16508, 'unique_tickers': 7314, 'unique_permnos': 6675, 'date_min': '2008-06-27T00:00:00+00:00', 'date_max': '2023-09-28T00:00:00+00:00', 'unique_authors': 'N/A', 'unique_threads': 'N/A', 'source': 'c:\\Users\\eagle\\RedditStock\\Data\\process\\00_biild_ticker_list\\2026-10-05_08-12-26_UTC\\Ticker_name.txt'}


## Matching engine: Token + Hash Set

Use Unicode word tokens, uppercase matching and a frozen ticker set. Repeated mentions of a ticker within one comment count once.

01 controls the input sample, including exclusion of UTC months 2012-04 and 2020-08. Re-run 01, then all cells in 02 (including long-format output), then 02_01. Frequency rankings and counts must be regenerated for the new matching run. Review the new PERMNO-based del decisions before running 02_02; old review files have different source hashes and must not be copied wholesale.


In [2]:
TOKEN_RE = re.compile(r'\w+', re.UNICODE)

def make_token_matcher(words):
    words = frozenset(words)
    def match(text):
        return sorted({token for token in TOKEN_RE.findall(text.upper()) if token in words})
    return match

SELECTED_METHOD = 'token_hash_set'
match = make_token_matcher(TICKER_SET)

cases = {
    '$AAPL AAPL, AAPL.': ['AAPL'],
    'aapl / MsFt': ['AAPL', 'MSFT'],
    'XAAPL AAPLX AAPL2 2AAPL AAPL_test _AAPL': [],
    '中AAPL AAPLé': [],
    '($AAPL), MSFT. AAPL': ['AAPL', 'MSFT'],
    '': [], 'nothing here': [], 'AAPL\nMSFT': ['AAPL', 'MSFT'],
    'AA AAA AAAA': ['AAA'],
}
for factory in [make_token_matcher]:
    test_match = factory({'AAPL', 'MSFT', 'AAA'})
    for body, expected in cases.items():
        assert test_match(body) == expected, (factory.__name__, body, test_match(body))
print('Boundary, case, repeated mention, multi-ticker and empty-body checks passed.')


def make_date_resolver(history):
    intervals = {}
    for row in history.itertuples(index=False):
        ticker = row.ticker.strip().upper()
        start = pd.Timestamp(row.secinfostartdt).date().toordinal()
        end = pd.Timestamp(row.secinfoenddt).date().toordinal()
        permno = int(row.permno)
        if start > end or permno <= 0: raise ValueError('Invalid ticker history')
        intervals.setdefault(ticker, set()).add((start, end, permno))
    intervals = {t: tuple(sorted(rows)) for t, rows in intervals.items()}
    @lru_cache(maxsize=200_000)
    def resolve(ticker, ordinal):
        if ordinal is None: return ('unmatched', (), 'invalid_comment_date')
        candidates = tuple(sorted({permno for start, end, permno in intervals.get(ticker, ())
                                   if start <= ordinal <= end}))
        if not candidates: return ('unmatched', candidates, 'no_active_permno')
        if len(candidates) > 1: return ('ambiguous', candidates, 'multiple_active_permnos')
        return ('matched', candidates, '')
    return resolve

resolve_ticker_date = make_date_resolver(ticker_history)

# 起訖日、全天UTC、重複歷史列、不同PERMNO重疊與ticker重用。
test_history = pd.DataFrame([
    ('TEST', '2020-01-01', '2020-01-10', '1'),
    ('TEST', '2020-01-05', '2020-01-10', '1'),
    ('TEST', '2020-01-10', '2020-01-20', '2'),
], columns=['ticker', 'secinfostartdt', 'secinfoenddt', 'permno'])
test_resolve = make_date_resolver(test_history)
for day, status, ids in [('2019-12-31', 'unmatched', ()), ('2020-01-01', 'matched', (1,)),
                        ('2020-01-05', 'matched', (1,)), ('2020-01-10', 'ambiguous', (1, 2)),
                        ('2020-01-20', 'matched', (2,)), ('2020-01-21', 'unmatched', ())]:
    result = test_resolve('TEST', pd.Timestamp(day).date().toordinal())
    assert result[:2] == (status, ids), (day, result)
assert test_resolve('TEST', None) == ('unmatched', (), 'invalid_comment_date')
assert test_resolve('UNKNOWN', pd.Timestamp('2020-01-01').date().toordinal())[0] == 'unmatched'
print('Inclusive-date and distinct-PERMNO resolution checks passed.')


Boundary, case, repeated mention, multi-ticker and empty-body checks passed.
Inclusive-date and distinct-PERMNO resolution checks passed.


## Diagnostics helpers

Define date and row diagnostics used during production matching.


In [3]:
def date_info(series):
    dt = pd.to_datetime(pd.to_numeric(series, errors='coerce'), unit='s', utc=True, errors='coerce')
    return {'date_min': None if dt.dropna().empty else dt.min().isoformat(),
            'date_max': None if dt.dropna().empty else dt.max().isoformat(),
            'invalid_date_count': int(dt.isna().sum())}

def local_diagnostics(frame):
    return {'row_count': len(frame), **date_info(frame['created_utc']),
            'unique_authors': int(frame.loc[frame.author.ne(''), 'author'].nunique()),
            'unique_threads': int(frame.loc[frame.link_id.ne(''), 'link_id'].nunique()),
            'missing_comment_id_count': int(frame.id.eq('').sum())}


## Parquet 輸出

每個輸入檔會產生：

- `*.comments.parquet`
- `*.mentions.parquet`：僅 matched，包含 permno 與留言日期。
- `*.ticker_mapping_audit.parquet`：全部候選，包含 match_status、candidate_permnos 及 match_reason。

每處理完一個 chunk，就立即將匹配到的留言與 ticker mention 寫入 Parquet row group，並使用 Zstandard 壓縮。

- `matched_tickers` 為字串列表，`matched_permnos` 為等長 int64 列表，依位置一一對應。
- `created_utc` 保留原始字串格式。
- 寫入時先使用 `.partial` 暫存檔，成功關閉並驗證後才視為完成。
- 執行中斷則該次 run 不算完成，需在新目錄重新執行。
- 即使某輸入檔沒有任何匹配，也會建立有效的空 Parquet 檔。

## 統計

使用暫存 SQLite 計算每個輸入檔與各處理階段中的唯一值：

- 非空白 `author`
- 非空白 `thread`
- `ticker`

整體唯一值數量採跨檔案聯集計算，而不是直接加總各檔結果。

### 缺失 Comment ID

原始空白 comment ID 仍保留，並納入 diagnostics 統計。

兩種輸出都額外加入：

- `source_file`
- `source_row`：從 1 開始的原始 CSV 邏輯列號，不含 header

這兩個欄位只用於追溯來源，不會取代或虛構 Reddit ID。

In [4]:
def utc_datetimes(values):
    return pd.to_datetime(pd.to_numeric(values, errors='coerce'), unit='s', utc=True, errors='coerce').dt.as_unit('ns')

def thread_ids(values):
    normalized = values.fillna('').str.strip().str.lower().str.replace(r'^t3_', '', regex=True)
    # pandas 3 string dtype represents missing values as float NaN.
    # Use object dtype to retain real Python None for Arrow from_pylist.
    return normalized.astype(object).mask(normalized.eq(''), None)

test = utc_datetimes(pd.Series(['0', '1609459199', '1609459200', '0.5', 'bad', '']))
assert test.iloc[0].isoformat() == '1970-01-01T00:00:00+00:00'
assert test.iloc[1].strftime('%Y-%m') == '2020-12'
assert test.iloc[2].strftime('%Y-%m-%d') == '2021-01-01'
assert test.iloc[3].microsecond == 500000
assert test.iloc[4:].isna().all()
assert thread_ids(pd.Series(['t3_abc123', ' ABC123 ', '', None])).fillna('<null>').tolist() == ['abc123', 'abc123', '<null>', '<null>']
null_threads = thread_ids(pd.Series(['t3_abc123', '', 't3_', '   ', None], dtype=str))
assert null_threads.tolist() == ['abc123', None, None, None, None]
assert all(value is None for value in null_threads.iloc[1:])
assert pa.array(null_threads.tolist(), type=pa.string()).null_count == 4
print('UTC, thread-normalization and Arrow-null checks passed.')


COMMENT_SCHEMA = pa.schema([(c, pa.string()) for c in ['comment_id', 'author', 'created_utc', 'link_id', 'body']] +
                           [('matched_tickers', pa.list_(pa.string())), ('matched_permnos', pa.list_(pa.int64())), ('matched_ticker_count', pa.int32()), ('source_file', pa.string()), ('source_row', pa.int64())])
COMMENT_SCHEMA = pa.schema(list(COMMENT_SCHEMA) + [pa.field('id',pa.string()),pa.field('parent_id',pa.string()),pa.field('thread_id',pa.string()),pa.field('created_datetime_utc',pa.timestamp('ns',tz='UTC')),pa.field('created_date',pa.date32()),pa.field('year_month',pa.string())])
MENTION_SCHEMA = pa.schema([
    ('comment_id', pa.string()), ('ticker', pa.string()), ('permno', pa.int64()),
    ('source_file', pa.string()), ('source_row', pa.int64()), ('thread_id', pa.string()),
    ('created_datetime_utc', pa.timestamp('ns', tz='UTC')), ('created_date', pa.date32()),
    ('year_month', pa.string()), ('match_status', pa.string()),
])
AUDIT_SCHEMA = pa.schema([
    ('comment_id', pa.string()), ('ticker', pa.string()), ('permno', pa.int64()),
    ('source_file', pa.string()), ('source_row', pa.int64()), ('thread_id', pa.string()),
    ('created_datetime_utc', pa.timestamp('ns', tz='UTC')), ('created_date', pa.date32()),
    ('year_month', pa.string()), ('match_status', pa.string()),
    ('candidate_permnos', pa.list_(pa.int64())), ('match_reason', pa.string()),
])
WORK_DB = RUN_DIR/'_work/diagnostics.sqlite'
WORK_DB.parent.mkdir()
db = sqlite3.connect(WORK_DB)
db.execute('PRAGMA cache_size=-32768')
db.execute('PRAGMA temp_store=FILE')
db.execute('CREATE TABLE uniques (file TEXT, stage TEXT, kind TEXT, value TEXT, PRIMARY KEY(file,stage,kind,value)) WITHOUT ROWID')

def record_uniques(file, stage, frame, ticker_values=()):
    for kind, values in [('authors', frame.author.unique()), ('threads', frame.link_id.unique()), ('tickers', ticker_values)]:
        db.executemany('INSERT OR IGNORE INTO uniques VALUES (?,?,?,?)',
                       ((file, stage, kind, str(value)) for value in values if value != ''))

def unique_count(stage, kind, file=None):
    if file is None:
        return db.execute('SELECT COUNT(DISTINCT value) FROM uniques WHERE stage=? AND kind=?', (stage, kind)).fetchone()[0]
    return db.execute('SELECT COUNT(*) FROM uniques WHERE file=? AND stage=? AND kind=?', (file, stage, kind)).fetchone()[0]

def merge_dates(stats, info, prefix):
    for suffix, func in [('date_min', min), ('date_max', max)]:
        key = prefix + suffix
        vals = [x for x in [stats.get(key), info[suffix]] if x is not None]
        stats[key] = func(vals) if vals else None
    key = prefix + 'invalid_date_count'
    stats[key] = stats.get(key, 0) + info['invalid_date_count']

def process_file(source):
    started = time.perf_counter()
    stats = {'file': source.name, 'raw_row_count': 0, 'matched_comment_count': 0,
             'comments_with_ge2_tickers': 0, 'mention_row_count': 0,
             'raw_missing_comment_id_count': 0, 'matched_missing_comment_id_count': 0,
             'candidate_mention_count': 0, 'unmatched_mention_count': 0,
             'ambiguous_mention_count': 0, 'invalid_date_candidate_count': 0,
             'lexical_candidate_comment_count': 0, 'comments_losing_all_candidates': 0}
    comments_path = RUN_DIR / (source.stem + '.comments.parquet')
    mentions_path = RUN_DIR / (source.stem + '.mentions.parquet')
    audit_path = RUN_DIR / (source.stem + '.ticker_mapping_audit.parquet')
    cp, mp, ap = [p.with_suffix('.parquet.partial') for p in [comments_path, mentions_path, audit_path]]
    with pq.ParquetWriter(cp, COMMENT_SCHEMA, compression='zstd') as cw, \
         pq.ParquetWriter(mp, MENTION_SCHEMA, compression='zstd') as mw, \
         pq.ParquetWriter(ap, AUDIT_SCHEMA, compression='zstd') as aw:
        for chunk_no, chunk in enumerate(read_chunks(source), 1):
            chunk_started = time.perf_counter()
            hits = chunk.body.map(match)
            candidate_mask = hits.map(bool)
            candidates = chunk.loc[candidate_mask].copy()
            candidates['source_file'] = source.name
            candidates['source_row'] = candidates.index.to_numpy() + 1
            candidates['thread_id'] = thread_ids(candidates.link_id)
            candidates['created_datetime_utc'] = utc_datetimes(candidates.created_utc)
            candidates['created_date'] = candidates.created_datetime_utc.dt.date
            candidates['year_month'] = candidates.created_datetime_utc.dt.strftime('%Y-%m')
            candidates['candidate_tickers'] = hits.loc[candidate_mask]
            candidates['matched_tickers'] = [[] for _ in range(len(candidates))]
            candidates['matched_permnos'] = [[] for _ in range(len(candidates))]
            audit_rows = []
            cols = ['id', 'candidate_tickers', 'source_row', 'thread_id', 'created_datetime_utc', 'created_date', 'year_month']
            for position, (comment_id, tickers, row, thread, timestamp, date, month) in enumerate(candidates[cols].itertuples(index=False, name=None)):
                valid_date = not pd.isna(date)
                ordinal = date.toordinal() if valid_date else None
                kept_tickers, kept_permnos = [], []
                for ticker in tickers:
                    status, ids, reason = resolve_ticker_date(ticker, ordinal)
                    permno = ids[0] if status == 'matched' else None
                    audit_rows.append({'comment_id': comment_id, 'ticker': ticker, 'permno': permno,
                        'source_file': source.name, 'source_row': int(row),
                        'thread_id': None if pd.isna(thread) else thread,
                        'created_datetime_utc': timestamp if valid_date else None,
                        'created_date': date if valid_date else None, 'year_month': month if valid_date else None,
                        'match_status': status, 'candidate_permnos': list(ids), 'match_reason': reason})
                    if status == 'matched':
                        kept_tickers.append(ticker); kept_permnos.append(permno)
                candidates.iat[position, candidates.columns.get_loc('matched_tickers')] = kept_tickers
                candidates.iat[position, candidates.columns.get_loc('matched_permnos')] = kept_permnos
            status_counts = Counter(row['match_status'] for row in audit_rows)
            stats['candidate_mention_count'] += len(audit_rows)
            stats['unmatched_mention_count'] += status_counts['unmatched']
            stats['ambiguous_mention_count'] += status_counts['ambiguous']
            stats['invalid_date_candidate_count'] += sum(row['match_reason'] == 'invalid_comment_date' for row in audit_rows)
            stats['lexical_candidate_comment_count'] += len(candidates)
            matched = candidates.loc[candidates.matched_tickers.map(bool)].copy()
            stats['comments_losing_all_candidates'] += len(candidates) - len(matched)
            matched['matched_ticker_count'] = matched.matched_tickers.map(len)
            chunk_tickers = {t for values in matched.matched_tickers for t in values}
            raw_diag, matched_diag = local_diagnostics(chunk), local_diagnostics(matched)
            stats['raw_missing_comment_id_count'] += raw_diag['missing_comment_id_count']
            stats['matched_missing_comment_id_count'] += matched_diag['missing_comment_id_count']
            stats['raw_row_count'] += len(chunk)
            stats['matched_comment_count'] += len(matched)
            stats['comments_with_ge2_tickers'] += int(matched.matched_ticker_count.ge(2).sum())
            merge_dates(stats, raw_diag, 'raw_'); merge_dates(stats, matched_diag, 'matched_')
            record_uniques(source.name, 'raw', chunk)
            record_uniques(source.name, 'matched', matched, chunk_tickers)
            db.commit()
            if audit_rows:
                audit = pa.Table.from_pylist(audit_rows, schema=AUDIT_SCHEMA)
                aw.write_table(audit, row_group_size=CHUNK_SIZE)
                successful = [row for row in audit_rows if row['match_status'] == 'matched']
                if successful:
                    mw.write_table(pa.Table.from_pylist(successful, schema=MENTION_SCHEMA), row_group_size=CHUNK_SIZE)
                stats['mention_row_count'] += len(successful)
                assert len(successful) == int(matched.matched_ticker_count.sum())
            if len(matched):
                output = matched.copy(); output['comment_id'] = output['id']
                assert all(len(ts) == len(ps) for ts, ps in zip(output.matched_tickers, output.matched_permnos))
                cw.write_table(pa.Table.from_pandas(output, schema=COMMENT_SCHEMA, preserve_index=False), row_group_size=CHUNK_SIZE)
            print({'file': source.name, 'chunk': chunk_no, 'raw': raw_diag, 'matched': matched_diag,
                   'candidate_status_counts': dict(status_counts), 'seconds': round(time.perf_counter() - chunk_started, 3)}, flush=True)
    assert pq.read_metadata(cp).num_rows == stats['matched_comment_count']
    assert pq.read_metadata(mp).num_rows == stats['mention_row_count']
    assert pq.read_metadata(ap).num_rows == stats['candidate_mention_count']
    assert stats['candidate_mention_count'] == stats['mention_row_count'] + stats['unmatched_mention_count'] + stats['ambiguous_mention_count']
    assert stats['lexical_candidate_comment_count'] == stats['matched_comment_count'] + stats['comments_losing_all_candidates']
    cp.replace(comments_path); mp.replace(mentions_path); ap.replace(audit_path)
    stats['match_rate'] = stats['matched_comment_count'] / stats['raw_row_count'] if stats['raw_row_count'] else 0.0
    for stage in ['raw', 'matched']:
        for kind in ['authors', 'threads']:
            stats[f'unique_{stage}_{kind}'] = unique_count(stage, kind, source.name)
    stats['unique_matched_tickers'] = unique_count('matched', 'tickers', source.name)
    stats['processing_seconds'] = time.perf_counter() - started
    display(pd.DataFrame([stats]))
    return stats


UTC, thread-normalization and Arrow-null checks passed.


## Process all input files

Token/hash-set matching processes every cleaned input row. Per-chunk diagnostics and per-file summaries are recorded.


In [5]:
formal_started = time.perf_counter()
summaries = []
for source in SOURCES:
    print('Processing:', source.name, flush=True)
    summaries.append(process_file(source))
formal_seconds = time.perf_counter() - formal_started


Processing: wallstreetbets_comment_part1.csv
{'file': 'wallstreetbets_comment_part1.csv', 'chunk': 1, 'raw': {'row_count': 50000, 'date_min': '1970-01-01T00:00:01+00:00', 'date_max': '2015-05-12T05:47:24+00:00', 'invalid_date_count': 0, 'unique_authors': 4591, 'unique_threads': 4475, 'missing_comment_id_count': 0}, 'matched': {'row_count': 29114, 'date_min': '2012-05-01T02:49:01+00:00', 'date_max': '2015-05-12T05:47:24+00:00', 'invalid_date_count': 0, 'unique_authors': 3562, 'unique_threads': 4126, 'missing_comment_id_count': 0}, 'candidate_status_counts': {'matched': 64915, 'unmatched': 47547, 'ambiguous': 73}, 'seconds': 2.637}
{'file': 'wallstreetbets_comment_part1.csv', 'chunk': 2, 'raw': {'row_count': 50000, 'date_min': '1970-01-01T00:00:03+00:00', 'date_max': '2015-09-02T15:49:31+00:00', 'invalid_date_count': 0, 'unique_authors': 3773, 'unique_threads': 3685, 'missing_comment_id_count': 0}, 'matched': {'row_count': 27308, 'date_min': '2015-05-12T06:50:40+00:00', 'date_max': '2015

,file,raw_row_count,matched_comment_count,comments_with_ge2_tickers,mention_row_count,raw_missing_comment_id_count,matched_missing_comment_id_count,candidate_mention_count,unmatched_mention_count,ambiguous_mention_count,...,matched_date_min,matched_date_max,matched_invalid_date_count,match_rate,unique_raw_authors,unique_raw_threads,unique_matched_authors,unique_matched_threads,unique_matched_tickers,processing_seconds
0,wallstreetbets_comment_part1.csv,299810,161556,79362,330802,0,0,551209,219988,419,...,2012-05-01T02:49:01+00:00,2016-04-25T12:28:09+00:00,0,0.538861,16570,20159,12673,18469,2006,17.72081


Processing: wallstreetbets_comment_part2.csv
{'file': 'wallstreetbets_comment_part2.csv', 'chunk': 1, 'raw': {'row_count': 50000, 'date_min': '2016-04-25T12:29:23+00:00', 'date_max': '2016-06-07T04:19:27+00:00', 'invalid_date_count': 0, 'unique_authors': 5051, 'unique_threads': 3081, 'missing_comment_id_count': 0}, 'matched': {'row_count': 25418, 'date_min': '2016-04-25T12:29:23+00:00', 'date_max': '2016-06-07T04:19:27+00:00', 'invalid_date_count': 0, 'unique_authors': 3662, 'unique_threads': 2753, 'missing_comment_id_count': 0}, 'candidate_status_counts': {'matched': 48750, 'unmatched': 31838, 'ambiguous': 65}, 'seconds': 2.394}
{'file': 'wallstreetbets_comment_part2.csv', 'chunk': 2, 'raw': {'row_count': 50000, 'date_min': '2016-06-07T04:19:53+00:00', 'date_max': '2016-07-13T19:18:32+00:00', 'invalid_date_count': 0, 'unique_authors': 5522, 'unique_threads': 3442, 'missing_comment_id_count': 0}, 'matched': {'row_count': 24808, 'date_min': '2016-06-07T04:19:53+00:00', 'date_max': '2016

,file,raw_row_count,matched_comment_count,comments_with_ge2_tickers,mention_row_count,raw_missing_comment_id_count,matched_missing_comment_id_count,candidate_mention_count,unmatched_mention_count,ambiguous_mention_count,...,matched_date_min,matched_date_max,matched_invalid_date_count,match_rate,unique_raw_authors,unique_raw_threads,unique_matched_authors,unique_matched_threads,unique_matched_tickers,processing_seconds
0,wallstreetbets_comment_part2.csv,537839,261889,116793,485189,0,0,826596,340792,615,...,2016-04-25T12:29:23+00:00,2017-04-01T06:12:18+00:00,0,0.486928,31195,32703,21742,28909,1863,39.663823


Processing: wallstreetbets_comment_part3.csv
{'file': 'wallstreetbets_comment_part3.csv', 'chunk': 1, 'raw': {'row_count': 50000, 'date_min': '2017-04-01T05:41:35+00:00', 'date_max': '2017-05-01T03:36:16+00:00', 'invalid_date_count': 0, 'unique_authors': 6972, 'unique_threads': 2724, 'missing_comment_id_count': 0}, 'matched': {'row_count': 24766, 'date_min': '2017-04-01T05:47:17+00:00', 'date_max': '2017-05-01T03:35:32+00:00', 'invalid_date_count': 0, 'unique_authors': 4663, 'unique_threads': 2399, 'missing_comment_id_count': 0}, 'candidate_status_counts': {'matched': 46262, 'unmatched': 31821, 'ambiguous': 50}, 'seconds': 2.432}
{'file': 'wallstreetbets_comment_part3.csv', 'chunk': 2, 'raw': {'row_count': 50000, 'date_min': '2017-05-01T03:36:22+00:00', 'date_max': '2017-06-01T02:49:43+00:00', 'invalid_date_count': 0, 'unique_authors': 6452, 'unique_threads': 2858, 'missing_comment_id_count': 0}, 'matched': {'row_count': 24430, 'date_min': '2017-05-01T03:36:22+00:00', 'date_max': '2017

...

,file,raw_row_count,matched_comment_count,comments_with_ge2_tickers,mention_row_count,raw_missing_comment_id_count,matched_missing_comment_id_count,candidate_mention_count,unmatched_mention_count,ambiguous_mention_count,...,matched_date_min,matched_date_max,matched_invalid_date_count,match_rate,unique_raw_authors,unique_raw_threads,unique_matched_authors,unique_matched_threads,unique_matched_tickers,processing_seconds
0,wallstreetbets_comment_part4.csv,3470434,1591538,619773,2656448,0,0,4556408,1897765,2195,...,2018-12-31T00:00:33+00:00,2020-02-01T00:00:15+00:00,0,0.458599,178671,48459,96082,42667,2375,182.463549


Processing: wallstreetbets_comment_part5.csv
{'file': 'wallstreetbets_comment_part5.csv', 'chunk': 1, 'raw': {'row_count': 50000, 'date_min': '2020-02-01T00:00:02+00:00', 'date_max': '2020-02-04T00:31:54+00:00', 'invalid_date_count': 0, 'unique_authors': 7632, 'unique_threads': 804, 'missing_comment_id_count': 0}, 'matched': {'row_count': 22545, 'date_min': '2020-02-01T00:00:07+00:00', 'date_max': '2020-02-04T00:31:54+00:00', 'invalid_date_count': 0, 'unique_authors': 5023, 'unique_threads': 668, 'missing_comment_id_count': 0}, 'candidate_status_counts': {'unmatched': 26532, 'matched': 37250, 'ambiguous': 32}, 'seconds': 2.354}
{'file': 'wallstreetbets_comment_part5.csv', 'chunk': 2, 'raw': {'row_count': 50000, 'date_min': '2020-02-04T00:31:57+00:00', 'date_max': '2020-02-05T15:32:10+00:00', 'invalid_date_count': 0, 'unique_authors': 9355, 'unique_threads': 831, 'missing_comment_id_count': 0}, 'matched': {'row_count': 22906, 'date_min': '2020-02-04T00:32:01+00:00', 'date_max': '2020-02

,file,raw_row_count,matched_comment_count,comments_with_ge2_tickers,mention_row_count,raw_missing_comment_id_count,matched_missing_comment_id_count,candidate_mention_count,unmatched_mention_count,ambiguous_mention_count,...,matched_date_min,matched_date_max,matched_invalid_date_count,match_rate,unique_raw_authors,unique_raw_threads,unique_matched_authors,unique_matched_threads,unique_matched_tickers,processing_seconds
0,wallstreetbets_comment_part5.csv,4285821,1917427,730855,3147879,0,0,5333284,2183548,1857,...,2020-02-01T00:00:07+00:00,2020-06-01T00:00:04+00:00,0,0.447388,159494,52454,108704,47353,2166,223.276823


Processing: wallstreetbets_comment_part6.csv
{'file': 'wallstreetbets_comment_part6.csv', 'chunk': 1, 'raw': {'row_count': 50000, 'date_min': '2020-06-01T00:00:01+00:00', 'date_max': '2020-06-02T06:02:08+00:00', 'invalid_date_count': 0, 'unique_authors': 9467, 'unique_threads': 1035, 'missing_comment_id_count': 0}, 'matched': {'row_count': 23118, 'date_min': '2020-06-01T00:00:04+00:00', 'date_max': '2020-06-02T06:02:08+00:00', 'invalid_date_count': 0, 'unique_authors': 6325, 'unique_threads': 914, 'missing_comment_id_count': 0}, 'candidate_status_counts': {'matched': 36811, 'unmatched': 24666, 'ambiguous': 14}, 'seconds': 2.479}
{'file': 'wallstreetbets_comment_part6.csv', 'chunk': 2, 'raw': {'row_count': 50000, 'date_min': '2020-06-02T06:02:13+00:00', 'date_max': '2020-06-03T14:10:44+00:00', 'invalid_date_count': 0, 'unique_authors': 9794, 'unique_threads': 954, 'missing_comment_id_count': 0}, 'matched': {'row_count': 23839, 'date_min': '2020-06-02T06:02:18+00:00', 'date_max': '2020-0

,file,raw_row_count,matched_comment_count,comments_with_ge2_tickers,mention_row_count,raw_missing_comment_id_count,matched_missing_comment_id_count,candidate_mention_count,unmatched_mention_count,ambiguous_mention_count,...,matched_date_min,matched_date_max,matched_invalid_date_count,match_rate,unique_raw_authors,unique_raw_threads,unique_matched_authors,unique_matched_threads,unique_matched_tickers,processing_seconds
0,wallstreetbets_comment_part6.csv,2191177,1018392,393737,1686629,0,0,2790930,1103236,1065,...,2020-06-01T00:00:04+00:00,2020-07-31T23:59:59+00:00,0,0.464769,93335,21787,63857,19055,2091,118.912993


## Combined statistics and output validation
Match rate is total matched input rows divided by total input rows. Unique authors, threads and tickers use exact unions across the six files. Parquet footer counts validate incremental writing without loading the output dataset into memory. A small preview reads only the first batch.

In [6]:
overall = {'file': 'ALL_FILES', 'processing_seconds': formal_seconds}
for key in ['raw_row_count', 'matched_comment_count', 'comments_with_ge2_tickers', 'mention_row_count',
            'raw_invalid_date_count', 'matched_invalid_date_count', 'raw_missing_comment_id_count', 'matched_missing_comment_id_count',
            'candidate_mention_count', 'unmatched_mention_count', 'ambiguous_mention_count', 'invalid_date_candidate_count',
            'lexical_candidate_comment_count', 'comments_losing_all_candidates']:
    overall[key] = sum(s.get(key, 0) for s in summaries)
overall['match_rate'] = overall['matched_comment_count'] / overall['raw_row_count'] if overall['raw_row_count'] else 0.0
for stage in ['raw', 'matched']:
    for suffix, func in [('date_min', min), ('date_max', max)]:
        vals = [s[stage + '_' + suffix] for s in summaries if s.get(stage + '_' + suffix)]
        overall[stage + '_' + suffix] = func(vals) if vals else None
    for kind in ['authors', 'threads']:
        overall[f'unique_{stage}_{kind}'] = unique_count(stage, kind)
overall['unique_matched_tickers'] = unique_count('matched', 'tickers')
summary_table = pd.DataFrame(summaries + [overall])
display(summary_table)
assert INPUT_STATS == {str(p): (p.stat().st_size, p.stat().st_mtime_ns) for p in SOURCES}, 'Input file metadata changed'
assert sum(pq.read_metadata(RUN_DIR / (s.stem + '.comments.parquet')).num_rows for s in SOURCES) == overall['matched_comment_count']
assert sum(pq.read_metadata(RUN_DIR / (s.stem + '.mentions.parquet')).num_rows for s in SOURCES) == overall['mention_row_count']
manifest = {'status': 'complete', 'schema_version':3, 'comment_enrichment':True, 'selected_method': SELECTED_METHOD, 'chunk_size': CHUNK_SIZE,
            'dictionary': str(DICTIONARY),
            'date_matching_policy':'inclusive_UTC_comment_date_unique_active_PERMNO',
            'mapping_statuses':['matched','unmatched','ambiguous'],
            'invalid_date_policy':'unmatched_invalid_comment_date',
            'duplicate_history_policy':'distinct_active_PERMNOs_not_history_row_count',
            'mapping_audit_files':[s.stem+'.ticker_mapping_audit.parquet' for s in SOURCES],'ticker_history_rows':len(ticker_history),'dictionary_unique_tickers':len(TICKER_SET),
            'permno_file':str(PERMNO_FILE),'permno_sha256':hashlib.sha256(PERMNO_FILE.read_bytes()).hexdigest(),
            'ticker_source_summary':TICKER_SUMMARY,
            'dictionary_sha256': hashlib.sha256(DICTIONARY.read_bytes()).hexdigest(),
            'input_metadata': INPUT_STATS, 'versions': VERSIONS, 'python': platform.python_version(),
            'case_policy': 'uppercase text and dictionary', 'token_policy': 'Unicode word tokens',
            'created_utc_output_type': 'original string', 'run_directory': str(RUN_DIR)}
assert DICTIONARY.read_bytes() == DICTIONARY_BYTES and PERMNO_FILE.read_bytes() == PERMNO_BYTES, 'Ticker inputs changed'
assert overall['candidate_mention_count'] == overall['mention_row_count'] + overall['unmatched_mention_count'] + overall['ambiguous_mention_count']
assert sum(pq.read_metadata(RUN_DIR / (s.stem + '.ticker_mapping_audit.parquet')).num_rows for s in SOURCES) == overall['candidate_mention_count']
db.close()
remove_work_database(WORK_DB)
for source in SOURCES:
    preview = next(pq.ParquetFile(RUN_DIR / (source.stem + '.comments.parquet')).iter_batches(batch_size=3), None)
    if preview is not None:
        display(preview.to_pandas())
        break
print('COMPLETE. Input sizes/mtimes unchanged; all Parquet row counts validated.')
print('Output:', RUN_DIR)

publish(ROOT,'matching',RUN_DIR,dict(manifest,files=summaries,overall=overall))


,file,raw_row_count,matched_comment_count,comments_with_ge2_tickers,mention_row_count,raw_missing_comment_id_count,matched_missing_comment_id_count,candidate_mention_count,unmatched_mention_count,ambiguous_mention_count,...,matched_date_min,matched_date_max,matched_invalid_date_count,match_rate,unique_raw_authors,unique_raw_threads,unique_matched_authors,unique_matched_threads,unique_matched_tickers,processing_seconds
0,wallstreetbets_comment_part1.csv,299810,161556,79362,330802,0,0,551209,219988,419,...,2012-05-01T02:49:01+00:00,2016-04-25T12:28:09+00:00,0,0.538861,16570,20159,12673,18469,2006,17.720810
1,wallstreetbets_comment_part2.csv,537839,261889,116793,485189,0,0,826596,340792,615,...,2016-04-25T12:29:23+00:00,2017-04-01T06:12:18+00:00,0,0.486928,31195,32703,21742,28909,1863,39.663823
2,wallstreetbets_comment_part3.csv,2097094,1000641,415116,1742820,0,0,2991047,1246718,1509,...,2017-04-01T05:47:17+00:00,2018-12-31T00:05:03+00:00,0,0.477156,95125,97121,64818,86950,2326,114.239498
3,wallstreetbets_comment_part4.csv,3470434,1591538,619773,2656448,0,0,4556408,1897765,2195,...,2018-12-31T00:00:33+00:00,2020-02-01T00:00:15+00:00,0,0.458599,178671,48459,96082,42667,2375,182.463549
4,wallstreetbets_comment_part5.csv,4285821,1917427,730855,3147879,0,0,5333284,2183548,1857,...,2020-02-01T00:00:07+00:00,2020-06-01T00:00:04+00:00,0,0.447388,159494,52454,108704,47353,2166,223.276823
5,wallstreetbets_comment_part6.csv,2191177,1018392,393737,1686629,0,0,2790930,1103236,1065,...,2020-06-01T00:00:04+00:00,2020-07-31T23:59:59+00:00,0,0.464769,93335,21787,63857,19055,2091,118.912993
6,ALL_FILES,12882175,5951443,2355636,10049767,0,0,17049474,6992047,7660,...,2012-05-01T02:49:01+00:00,2020-07-31T23:59:59+00:00,0,0.461991,406172,269646,259596,241473,3936,696.632422


,comment_id,author,created_utc,link_id,body,matched_tickers,matched_permnos,matched_ticker_count,source_file,source_row,id,parent_id,thread_id,created_datetime_utc,created_date,year_month
0,c4il4l6,dustybus,1335840541.0,t3_t0olb,Thanks for the awesome reply,[FOR],[92478],1,wallstreetbets_comment_part1.csv,1,c4il4l6,t1_c4ikmnk,t0olb,2012-05-01 02:49:01+00:00,2012-05-01,2012-05
1,c4imwtq,simplemethod,1335848878.0,t3_t0olb,I am personally bearish on Netflix I did like ...,"[AWAY, DVD, FOR, LOOK, LOW, POST]","[12882, 84026, 92478, 87173, 61399, 13188]",6,wallstreetbets_comment_part1.csv,2,c4imwtq,t3_t0olb,t0olb,2012-05-01 05:07:58+00:00,2012-05-01,2012-05
2,c4iq6sd,hindermore,1335879367.0,t3_suslt,I guess the earnings werent super impressive o...,[OPEN],[92934],1,wallstreetbets_comment_part1.csv,3,c4iq6sd,t3_suslt,suslt,2012-05-01 13:36:07+00:00,2012-05-01,2012-05


COMPLETE. Input sizes/mtimes unchanged; all Parquet row counts validated.
Output: c:\Users\eagle\RedditStock\Data\process\02_ticker_matching\2026-10-15_02-43-43_UTC


## comment–ticker–PERMNO long format
直接串接本次 `.mentions.parquet` 的成功配對，保留 `permno`、留言 UTC 日期、月份、thread 及來源列號。不重新以 ticker 猜測 PERMNO，不去重、不納入 unmatched／ambiguous。

只接受 schema version 3 以上且具日期對應規則的 run。舊版輸出缺少時間點 PERMNO，須重跑 02 才能產生此格式。


In [7]:
from pathlib import Path
import json
import pyarrow as pa
import pyarrow.parquet as pq

COMMENT_TICKER_INPUT = None
ct_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'Data/process').is_dir())
if COMMENT_TICKER_INPUT is not None:
    ct_run = Path(COMMENT_TICKER_INPUT)
elif 'RUN_DIR' in globals():
    ct_run = Path(RUN_DIR)
else:
    ct_run = Path(json.loads((ct_root/'Data/process/pipeline_runs.json').read_text())['matching_manifest']).parent
ct_summary = json.loads((ct_run/'run_summary.json').read_text(encoding='utf-8'))
assert ct_summary['status'] == 'complete' and ct_summary.get('schema_version', 0) >= 3
assert ct_summary['date_matching_policy'] == 'inclusive_UTC_comment_date_unique_active_PERMNO'
ct_sources = sorted(ct_run.glob('*.mentions.parquet'))
assert ct_sources
ct_input_stats = {str(p): (p.stat().st_size, p.stat().st_mtime_ns) for p in ct_sources}
ct_schema = pa.schema([(c, pa.string()) for c in ['year_month','thread_id','comment_id','ticker','source_file']] +
                     [('source_row', pa.int64()), ('permno', pa.int64()), ('created_date', pa.date32()),
                      ('created_datetime_utc', pa.timestamp('ns', tz='UTC')), ('match_status', pa.string())])
ct_dest = ct_run/'comment_ticker_long.parquet'
ct_tmp = ct_dest.with_suffix('.parquet.partial')
ct_rows = 0
with pq.ParquetWriter(ct_tmp, ct_schema, compression='zstd') as ct_writer:
    for source in ct_sources:
        for batch in pq.ParquetFile(source).iter_batches(batch_size=50_000, columns=ct_schema.names):
            frame = batch.to_pandas()
            assert frame.match_status.eq('matched').all() and frame.permno.notna().all()
            assert frame.permno.gt(0).all() and frame.created_date.notna().all()
            times = pd.to_datetime(frame.created_datetime_utc, utc=True)
            assert times.dt.strftime('%Y-%m').equals(frame.year_month)
            assert (times.dt.date == frame.created_date).all()
            assert not frame.duplicated(['source_file','source_row','ticker']).any()
            ct_writer.write_table(pa.Table.from_pandas(frame, schema=ct_schema, preserve_index=False), row_group_size=50_000)
            ct_rows += len(frame)
assert ct_rows == pq.read_metadata(ct_tmp).num_rows == ct_summary['overall']['mention_row_count']
assert ct_input_stats == {str(p): (p.stat().st_size, p.stat().st_mtime_ns) for p in ct_sources}
ct_tmp.replace(ct_dest)
print('成功配對 long-format 列數：', ct_rows, '；輸出：', ct_dest)
ct_preview = next(pq.ParquetFile(ct_dest).iter_batches(batch_size=10), None)
if ct_preview is not None: display(ct_preview.to_pandas())


成功配對 long-format 列數： 10049767 ；輸出： c:\Users\eagle\RedditStock\Data\process\02_ticker_matching\2026-10-15_02-43-43_UTC\comment_ticker_long.parquet


,year_month,thread_id,comment_id,ticker,source_file,source_row,permno,created_date,created_datetime_utc,match_status
0,2012-05,t0olb,c4il4l6,FOR,wallstreetbets_comment_part1.csv,1,92478,2012-05-01,2012-05-01 02:49:01+00:00,matched
1,2012-05,t0olb,c4imwtq,AWAY,wallstreetbets_comment_part1.csv,2,12882,2012-05-01,2012-05-01 05:07:58+00:00,matched
2,2012-05,t0olb,c4imwtq,DVD,wallstreetbets_comment_part1.csv,2,84026,2012-05-01,2012-05-01 05:07:58+00:00,matched
3,2012-05,t0olb,c4imwtq,FOR,wallstreetbets_comment_part1.csv,2,92478,2012-05-01,2012-05-01 05:07:58+00:00,matched
4,2012-05,t0olb,c4imwtq,LOOK,wallstreetbets_comment_part1.csv,2,87173,2012-05-01,2012-05-01 05:07:58+00:00,matched
5,2012-05,t0olb,c4imwtq,LOW,wallstreetbets_comment_part1.csv,2,61399,2012-05-01,2012-05-01 05:07:58+00:00,matched
6,2012-05,t0olb,c4imwtq,POST,wallstreetbets_comment_part1.csv,2,13188,2012-05-01,2012-05-01 05:07:58+00:00,matched
7,2012-05,suslt,c4iq6sd,OPEN,wallstreetbets_comment_part1.csv,3,92934,2012-05-01,2012-05-01 13:36:07+00:00,matched
8,2012-05,t0olb,c4iqjuz,FOR,wallstreetbets_comment_part1.csv,4,92478,2012-05-01,2012-05-01 14:22:35+00:00,matched
9,2012-05,t0olb,c4iqjuz,NFLX,wallstreetbets_comment_part1.csv,4,89393,2012-05-01,2012-05-01 14:22:35+00:00,matched
